In [1]:
import pandas as pd
import tensorflow as tf

In [41]:
df = pd.read_csv('kaggle_sentiment/tweet_sentiment_train.csv', encoding='utf-8', encoding_errors='replace')
df = df.dropna(subset=["text", "sentiment"]).reset_index(drop=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27480 entries, 0 to 27479
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   textID            27480 non-null  str    
 1   text              27480 non-null  str    
 2   selected_text     27480 non-null  str    
 3   sentiment         27480 non-null  str    
 4   Time of Tweet     27480 non-null  str    
 5   Age of User       27480 non-null  str    
 6   Country           27480 non-null  str    
 7   Population -2020  27480 non-null  int64  
 8   Land Area (Km�)   27480 non-null  float64
 9   Density (P/Km�)   27480 non-null  int64  
dtypes: float64(1), int64(2), str(7)
memory usage: 5.8 MB


In [32]:
df[:3]

,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km�),Density (P/Km�)
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral,morning,0-20,Afghanistan,38928346,652860.0,60
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400.0,105
2,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740.0,18


In [42]:
df['text'] = df['text'].astype(str)

In [43]:
sentiment_mapping = {'negative': 2, 'neutral': 0, 'positive': 1}
df['sentiment'] = df['sentiment'].replace(sentiment_mapping)
df['sentiment'] = df['sentiment'].astype(float)
df[['text', 'sentiment']].head(10)

,text,sentiment
0,"I`d have responded, if I were going",0.0
1,Sooo SAD I will miss you here in San Diego!!!,2.0
2,my boss is bullying me...,2.0
3,what interview! leave me alone,2.0
4,"Sons of ****, why couldn`t they put them on t...",2.0
5,http://www.dothebouncy.com/smf - some shameles...,0.0
6,2am feedings for the baby are fun when he is a...,1.0
7,Soooo high,0.0
8,Both of you,0.0
9,Journey!? Wow... u just became cooler. hehe....,1.0


In [44]:
texts = df['text'].values
labels = tf.constant(df['sentiment'].values)
# Create the dataset
dataset = tf.data.Dataset.from_tensor_slices((texts, labels))
for x, y in dataset.take(1):
    print(x, y)

tf.Tensor(b' I`d have responded, if I were going', shape=(), dtype=string) tf.Tensor(0.0, shape=(), dtype=float64)


### Dataset bereinigen

In [45]:
import string
import re


def custom_standardization(input_data):
    lowercase = tf.strings.lower(input_data)
    stripped_html = tf.strings.regex_replace(lowercase, '<br />', ' ')
    return tf.strings.regex_replace(
            stripped_html,
            '[%s]' % re.escape(string.punctuation),
            ''
            )

In [46]:
dataset = dataset.map(lambda x, y: ((custom_standardization(x), y)))
for text, label in dataset.take(2):
    print(text.numpy())
    print(label.numpy())

b' id have responded if i were going'
0.0
b' sooo sad i will miss you here in san diego'
2.0


2026-06-26 07:58:01.608244: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [47]:
train_size = 22_000
val_size = 2_000
test_size = len(df) - train_size - val_size

dataset = dataset.shuffle(train_size + val_size)
train_ds = dataset.take(train_size)
val_ds = dataset.skip(train_size).take(val_size)
test_ds = dataset.skip(train_size + val_size)

tf.data.Dataset.save(train_ds, "kaggle_sentiment/train_ds")
tf.data.Dataset.save(val_ds, "kaggle_sentiment/val_ds")
tf.data.Dataset.save(test_ds, "kaggle_sentiment/test_ds")

In [17]:
train_ds = train_ds.batch(128).cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(128).cache().prefetch(tf.data.AUTOTUNE)

In [18]:
max_sequence_length = 0
for text, label in dataset:
    if len(text.numpy()) > max_sequence_length:
        max_sequence_length = len(text.numpy())
print(max_sequence_length)

143


## BERT-Classifier trainieren

In [19]:
import keras_hub

bert_name = "bert_tiny_en_uncased"

classifier = keras_hub.models.TextClassifier.from_preset(bert_name, sequence_lengths=256, num_classes=3)
classifier.build(input_shape=(None, 256))
classifier.summary()

Preprocessor: "bert_text_classifier_preprocessor_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ bert_tokenizer (BertTokenizer)                                │                       Vocab size: 30,522 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "bert_text_classifier_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ segment_ids (InputLayer)      │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bert_backbone (BertBackbone)  │ [(None, 128), (None,      │       4,385,920 │ padding_mask[0][0],        │
│                               │ None, 128)]               │                 │ segment_ids[0][0],         │
│                               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ classifier_dropout (Dropout)  │ (None, 128)               │               0 │ bert_backbone[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ logits (Dense)                │ (None, 3)                 │             387 │ classifier_dropout[0][0]   │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,386,307 (16.73 MB)

 Trainable params: 4,386,307 (16.73 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=2,
        min_delta=0.02,
        restore_best_weights=True
        )
classifier.fit(train_ds, epochs=50, validation_data=val_ds, verbose=1)
classifier.save("bert_sentiment_256.keras")

Epoch 1/20


/Users/thomasbayer/Projects/PycharmProjects/lecture/dl_summer/.venv_312/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


172/172 ━━━━━━━━━━━━━━━━━━━━ 43s 248ms/step - loss: 0.4471 - sparse_categorical_accuracy: 0.8265 - val_loss: 0.4345 - val_sparse_categorical_accuracy: 0.8455
Epoch 2/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 43s 250ms/step - loss: 0.4272 - sparse_categorical_accuracy: 0.8370 - val_loss: 0.4213 - val_sparse_categorical_accuracy: 0.8490
Epoch 3/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 41s 238ms/step - loss: 0.4041 - sparse_categorical_accuracy: 0.8457 - val_loss: 0.4057 - val_sparse_categorical_accuracy: 0.8575
Epoch 4/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 42s 244ms/step - loss: 0.3907 - sparse_categorical_accuracy: 0.8495 - val_loss: 0.3917 - val_sparse_categorical_accuracy: 0.8665
Epoch 5/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 46s 265ms/step - loss: 0.3673 - sparse_categorical_accuracy: 0.8633 - val_loss: 0.3797 - val_sparse_categorical_accuracy: 0.8740
Epoch 6/20
172/172 ━━━━━━━━━━━━━━━━━━━━ 57s 329ms/step - loss: 0.3478 - sparse_categorical_accuracy: 0.8731 - val_loss: 0.3707 - val_sparse_categorical_accuracy: 0.87

In [22]:
classifier.evaluate(test_ds.batch(16))

218/218 ━━━━━━━━━━━━━━━━━━━━ 5s 20ms/step - loss: 0.3022 - sparse_categorical_accuracy: 0.9158


[0.3021940290927887, 0.9158046245574951]

In [39]:
import tensorflow as tf
import keras
import keras_hub

bert_name = "bert_tiny_en_uncased"

preprocess_layer = keras_hub.models.BertPreprocessor.from_preset(
        bert_name,
        trainable=False,
        sequence_length=256,
        )

backbone = keras_hub.models.Backbone.from_preset(
        bert_name,
        trainable=False,
        sequence_length=256,
        )

text_input = keras.Input(shape=(), dtype=tf.string)

bert_inputs = preprocess_layer(text_input)
bert_outputs = backbone(bert_inputs)

x = bert_outputs["pooled_output"]
x = keras.layers.LayerNormalization()(x)
x = keras.layers.Dropout(0.2)(x)
x = keras.layers.Dense(128, activation="gelu")(x)
x = keras.layers.Dense(64, activation="gelu")(x)
x = keras.layers.Dense(16, activation="gelu")(x)
x = keras.layers.Dropout(0.2)(x)

logits = keras.layers.Dense(3)(x)

custom_classifier = keras.Model(text_input, logits, name="bert_sentiment_classifier")

loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

custom_classifier.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=3e-4, weight_decay=1e-4),
        loss=loss_fn,
        metrics=["accuracy"],
        )
custom_classifier.summary()
"""
Die Ergebnisse sind nicht besonders gut. Wenn im backbone trainable=True gesetzt wird, sind die Ergebnisse deutlich besser (auch weil mehr Parameter trainiert werden können).
"""


Model: "bert_sentiment_classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_8       │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_text_classifi… │ [(None, 256),     │          0 │ input_layer_8[0]… │
│ (BertTextClassifie… │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_backbone       │ [(None, 128),     │  4,385,920 │ bert_text_classi… │
│ (BertBackbone)      │ (None, 256, 128)] │            │ bert_text_classi… │
│                     │                   │            │ bert_text_classi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 128)       │        256 │ bert_backbone[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_31          │ (None, 128)       │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_39 (Dense)    │ (None, 128)       │     16,512 │ dropout_31[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_40 (Dense)    │ (None, 64)        │      8,256 │ dense_39[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_41 (Dense)    │ (None, 16)        │      1,040 │ dense_40[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_32          │ (None, 16)        │          0 │ dense_41[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_42 (Dense)    │ (None, 3)         │         51 │ dropout_32[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,412,035 (16.83 MB)

 Trainable params: 26,115 (102.01 KB)

 Non-trainable params: 4,385,920 (16.73 MB)

'\nDie Ergebnisse sind nicht besonders gut. Wenn im backbone trainable=True gesetzt wird, sind die Ergebnisse deutlich besser (auch weil mehr Parameter trainiert werden können).\n'

In [36]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=2,
        min_delta=0.01,
        restore_best_weights=True
        )
custom_classifier.fit(train_ds, epochs=50, validation_data=val_ds, callbacks=[early_stopping_cb], verbose=1)

Epoch 1/50
  3/172 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.4193 - loss: 1.0694 

/Users/thomasbayer/Projects/PycharmProjects/lecture/dl_summer/.venv_312/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


172/172 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.4159 - loss: 1.0749 - val_accuracy: 0.4700 - val_loss: 1.0456
Epoch 2/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.4330 - loss: 1.0608 - val_accuracy: 0.4830 - val_loss: 1.0407
Epoch 3/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.4408 - loss: 1.0534 - val_accuracy: 0.4895 - val_loss: 1.0343
Epoch 4/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.4478 - loss: 1.0470 - val_accuracy: 0.4905 - val_loss: 1.0282
Epoch 5/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.4555 - loss: 1.0446 - val_accuracy: 0.4915 - val_loss: 1.0274


In [37]:
backbone.trainable = True

custom_classifier.compile(
        optimizer=keras.optimizers.AdamW(learning_rate=2e-5, weight_decay=1e-5),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
        )
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=2,
        min_delta=0.05,
        restore_best_weights=True
        )
custom_classifier.fit(train_ds, epochs=50, validation_data=val_ds, callbacks=[early_stopping_cb], verbose=1)
custom_classifier.save("bert_sentiment_custom_256.keras")

Epoch 1/50


/Users/thomasbayer/Projects/PycharmProjects/lecture/dl_summer/.venv_312/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


172/172 ━━━━━━━━━━━━━━━━━━━━ 30s 140ms/step - accuracy: 0.5021 - loss: 0.9966 - val_accuracy: 0.6040 - val_loss: 0.8832
Epoch 2/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 21s 124ms/step - accuracy: 0.5957 - loss: 0.8801 - val_accuracy: 0.6655 - val_loss: 0.7674
Epoch 3/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 21s 124ms/step - accuracy: 0.6463 - loss: 0.8035 - val_accuracy: 0.6980 - val_loss: 0.7190
Epoch 4/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 21s 125ms/step - accuracy: 0.6733 - loss: 0.7668 - val_accuracy: 0.7165 - val_loss: 0.6906
Epoch 5/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 22s 125ms/step - accuracy: 0.6916 - loss: 0.7320 - val_accuracy: 0.7250 - val_loss: 0.6660
Epoch 6/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 23s 133ms/step - accuracy: 0.7052 - loss: 0.7049 - val_accuracy: 0.7320 - val_loss: 0.6478
Epoch 7/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 26s 149ms/step - accuracy: 0.7118 - loss: 0.6899 - val_accuracy: 0.7370 - val_loss: 0.6334
Epoch 8/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 29s 171ms/step - accuracy: 0.7180 - loss: 0.6744 - val

KeyboardInterrupt: 